In [1]:
from pathlib import Path
import sys
import torch

current_dir = Path.cwd()

if (current_dir / "data").exists():
    PROJECT_ROOT = current_dir
else:
    PROJECT_ROOT = current_dir.parent

sys.path.append(str(PROJECT_ROOT))
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [2]:
from src.reid_model import PigReIDModel
num_classes = 6
model = PigReIDModel(num_classes=num_classes, embedding_dim=256, pretrained=True)
model = model.to(device)
model.eval()

dummy_images = torch.randn(4, 3, 224, 224).to(device)

with torch.no_grad():
    embeddings, logits = model(dummy_images)


print("Embeddings shape:", embeddings.shape)
print("Logits shape:", logits.shape)

Embeddings shape: torch.Size([4, 256])
Logits shape: torch.Size([4, 6])


In [3]:
from pathlib import Path
import sys
import pandas as pd
import torch

current_dir = Path.cwd()
if (current_dir / "data").exists():
    PROJECT_ROOT = current_dir
else:
    PROJECT_ROOT = current_dir.parent

sys.path.append(str(PROJECT_ROOT))

splits_path = PROJECT_ROOT / "data" / "metadata" / "splits.csv"

split_df = pd.read_csv(splits_path)

display(split_df.head())

print("Rows:", len(split_df))
display(split_df["pool_status"].value_counts())

,image_path,identity_id,original_path,file_name,split,pool_status,is_labeled
0,d:\farm_project\data\processed\pigreid_224\G12...,G12_54,d:\farm_project\data\raw\pigreid\G12_54\Basler...,Basler_acA4112-20uc__40331001__20240806_074832...,train_pool,initial_labeled,True
1,d:\farm_project\data\processed\pigreid_224\G12...,G12_54,d:\farm_project\data\raw\pigreid\G12_54\Basler...,Basler_acA4112-20uc__40331001__20240826_101022...,train_pool,initial_labeled,True
2,d:\farm_project\data\processed\pigreid_224\G12...,G12_54,d:\farm_project\data\raw\pigreid\G12_54\Basler...,Basler_acA4112-20uc__40331001__20240729_124640...,train_pool,initial_labeled,True
3,d:\farm_project\data\processed\pigreid_224\G12...,G12_54,d:\farm_project\data\raw\pigreid\G12_54\Basler...,Basler_acA4112-20uc__40331001__20240729_124640...,train_pool,initial_labeled,True
4,d:\farm_project\data\processed\pigreid_224\G12...,G12_54,d:\farm_project\data\raw\pigreid\G12_54\Basler...,Basler_acA4112-20uc__40331001__20240820_083924...,train_pool,initial_labeled,True


Rows: 480


pool_status
unlabeled          312
initial_labeled     72
gallery             72
query               24
Name: count, dtype: int64

In [4]:
identity_ids = sorted(split_df["identity_id"].unique())
identity_to_label = {identity_id: index for index, identity_id in enumerate(identity_ids)}
label_to_identity = {index: identity_id for identity_id, index in identity_to_label.items()}
split_df["label_idx"] = split_df["identity_id"].map(identity_to_label)

print("identity_to_label:", identity_to_label)

num_classes = len(identity_to_label)

print("num_classes:", num_classes)

display(split_df[["image_path", "identity_id", "label_idx", "pool_status"]].head())

identity_to_label: {'G12_54': 0, 'G12_57': 1, 'G12_61': 2, 'G12_62': 3, 'G12_66': 4, 'G8_78': 5}
num_classes: 6


,image_path,identity_id,label_idx,pool_status
0,d:\farm_project\data\processed\pigreid_224\G12...,G12_54,0,initial_labeled
1,d:\farm_project\data\processed\pigreid_224\G12...,G12_54,0,initial_labeled
2,d:\farm_project\data\processed\pigreid_224\G12...,G12_54,0,initial_labeled
3,d:\farm_project\data\processed\pigreid_224\G12...,G12_54,0,initial_labeled
4,d:\farm_project\data\processed\pigreid_224\G12...,G12_54,0,initial_labeled


In [5]:
from torchvision import transforms
from torch.utils.data import DataLoader
from src.dataset import PigReIDDataset


train_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


initial_labeled_df = split_df[split_df["pool_status"] == "initial_labeled"].copy()

initial_labeled_dataset = PigReIDDataset(
    dataframe=initial_labeled_df,
    transform=train_transform
)

initial_labeled_loader = DataLoader(
    initial_labeled_dataset,
    batch_size=16,
    shuffle=True,
    num_workers=0
)

batch = next(iter(initial_labeled_loader))

print("images shape:", batch["image"].shape)
print("labels shape:", batch["label"].shape)
print("labels:", batch["label"])
print("identity_ids:", batch["identity_id"])

images shape: torch.Size([16, 3, 224, 224])
labels shape: torch.Size([16])
labels: tensor([0, 4, 5, 5, 2, 3, 4, 3, 3, 1, 2, 1, 1, 4, 2, 4])
identity_ids: ['G12_54', 'G12_66', 'G8_78', 'G8_78', 'G12_61', 'G12_62', 'G12_66', 'G12_62', 'G12_62', 'G12_57', 'G12_61', 'G12_57', 'G12_57', 'G12_66', 'G12_61', 'G12_66']


In [6]:
from src.reid_model import PigReIDModel
import torch.nn as nn
from pytorch_metric_learning import losses

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = PigReIDModel(
    num_classes=num_classes,
    embedding_dim=256,
    pretrained=True
)

model = model.to(device)
ce_loss_fn = nn.CrossEntropyLoss()
triplet_loss_fn = losses.TripletMarginLoss(margin=0.3)

images = batch["image"]
labels = batch["label"]

images = images.to(device)
labels = labels.to(device)

embeddings, logits = model(images)

loss_ce = ce_loss_fn(logits, labels)
loss_triplet = triplet_loss_fn(embeddings, labels)
loss_total = loss_ce + loss_triplet

print("embeddings shape:", embeddings.shape)
print("logits shape:", logits.shape)
print("CE loss:", loss_ce.item())
print("Triplet loss:", loss_triplet.item())
print("Total loss:", loss_total.item())

d:\anaconda\Lib\site-packages\pytorch_metric_learning\utils\common_functions.py:9: UserWarning: A NumPy version >=1.23.5 and <2.5.0 is required for this version of SciPy (detected version 2.5.1)
  import scipy.stats


embeddings shape: torch.Size([16, 256])
logits shape: torch.Size([16, 6])
CE loss: 1.7910234928131104
Triplet loss: 0.3019152879714966
Total loss: 2.0929388999938965


In [7]:
from torch.optim import AdamW
from tqdm.auto import tqdm

NUM_EPOCHS = 10

LEARNING_RATE = 1e-4

model = PigReIDModel(
    num_classes=num_classes,
    embedding_dim=256,
    pretrained=True
)

model = model.to(device)

ce_loss_fn = nn.CrossEntropyLoss()

triplet_loss_fn = losses.TripletMarginLoss(margin=0.3)

optimizer = AdamW(
    model.parameters(),
    lr=LEARNING_RATE
)

train_history = []
for epoch in range(NUM_EPOCHS):
    model.train()
    total_loss_sum = 0.0
    ce_loss_sum = 0.0
    triplet_loss_sum = 0.0
    num_batches = 0
    
    for batch in tqdm(initial_labeled_loader, desc=f"Epoch {epoch + 1}/{NUM_EPOCHS}"):
        images = batch["image"]
        labels = batch["label"]
        images = images.to(device)
        labels = labels.to(device)
        optimizer.zero_grad()
        embeddings, logits = model(images)
        loss_ce = ce_loss_fn(logits, labels)
        loss_triplet = triplet_loss_fn(embeddings, labels)
        loss = loss_ce + loss_triplet
        loss.backward()
        optimizer.step()
        
        total_loss_sum += loss.item()
        ce_loss_sum += loss_ce.item()
        triplet_loss_sum += loss_triplet.item()
        num_batches += 1
    
    avg_total_loss = total_loss_sum / num_batches
    
    avg_ce_loss = ce_loss_sum / num_batches
    
    avg_triplet_loss = triplet_loss_sum / num_batches
    
    train_history.append({
        "epoch": epoch + 1,
        "total_loss": avg_total_loss,
        "ce_loss": avg_ce_loss,
        "triplet_loss": avg_triplet_loss
    })
    
    print(
        f"Epoch {epoch + 1}: "
        f"total_loss={avg_total_loss:.4f}, "
        f"ce_loss={avg_ce_loss:.4f}, "
        f"triplet_loss={avg_triplet_loss:.4f}"
    )

Epoch 1/10:   0%|          | 0/5 [00:00<?, ?it/s]

Epoch 1: total_loss=2.0925, ce_loss=1.7849, triplet_loss=0.3077


Epoch 2/10:   0%|          | 0/5 [00:00<?, ?it/s]

Epoch 2: total_loss=1.9678, ce_loss=1.7309, triplet_loss=0.2369


Epoch 3/10:   0%|          | 0/5 [00:00<?, ?it/s]

...

Epoch 8: total_loss=1.6107, ce_loss=1.5556, triplet_loss=0.0551


Epoch 9/10:   0%|          | 0/5 [00:00<?, ?it/s]

Epoch 9: total_loss=1.5644, ce_loss=1.5427, triplet_loss=0.0217


Epoch 10/10:   0%|          | 0/5 [00:00<?, ?it/s]

Epoch 10: total_loss=1.5706, ce_loss=1.5279, triplet_loss=0.0427


In [19]:
from src.evaluation import extract_embeddings, compute_rank1_map

query_df = split_df[split_df["pool_status"] == "query"].copy()
gallery_df = split_df[split_df["pool_status"] == "gallery"].copy()

query_dataset = PigReIDDataset(
    dataframe=query_df,
    transform=train_transform
)

gallery_dataset = PigReIDDataset(
    dataframe=gallery_df,
    transform=train_transform
)

query_loader = DataLoader(
    query_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0
)

gallery_loader = DataLoader(
    gallery_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0
)

query_embeddings, query_labels, query_identity_ids, query_image_paths = extract_embeddings(
    model=model,
    dataloader=query_loader,
    device=device
)

gallery_embeddings, gallery_labels, gallery_identity_ids, gallery_image_paths = extract_embeddings(
    model=model,
    dataloader=gallery_loader,
    device=device
)

metrics = compute_rank1_map(
    query_embeddings=query_embeddings,
    query_labels=query_labels,
    gallery_embeddings=gallery_embeddings,
    gallery_labels=gallery_labels
)

print(f"Rank-1: {metrics['rank1']:.4f}")
print(f"mAP: {metrics['mAP']:.4f}")

Rank-1: 0.1667
mAP: 0.2623


In [20]:
from src.active_learning import core_set_selection

labeled_df = split_df[split_df["pool_status"] == "initial_labeled"].copy()
unlabeled_df = split_df[split_df["pool_status"] == "unlabeled"].copy()

labeled_dataset = PigReIDDataset(
    dataframe=labeled_df,
    transform=train_transform
)

unlabeled_dataset = PigReIDDataset(
    dataframe=unlabeled_df,
    transform=train_transform
)

labeled_loader = DataLoader(
    labeled_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0
)

unlabeled_loader = DataLoader(
    unlabeled_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0
)


labeled_embeddings, labeled_labels, labeled_identity_ids, labeled_image_paths = extract_embeddings(
    model=model,
    dataloader=labeled_loader,
    device=device
)


unlabeled_embeddings, unlabeled_labels, unlabeled_identity_ids, unlabeled_image_paths = extract_embeddings(
    model=model,
    dataloader=unlabeled_loader,
    device=device
)

AL_BUDGET = 60


selected_indices = core_set_selection(
    labeled_embeddings=labeled_embeddings,
    unlabeled_embeddings=unlabeled_embeddings,
    budget=AL_BUDGET
)

selected_al_df = unlabeled_df.iloc[selected_indices].copy()

selected_al_df["pool_status"] = "active_labeled"

selected_al_df["is_labeled"] = True

print("Selected by AL:", len(selected_al_df))

display(selected_al_df["identity_id"].value_counts())

Selected by AL: 60


identity_id
G12_54    18
G12_61    17
G12_66    10
G12_62     8
G12_57     6
G8_78      1
Name: count, dtype: int64

In [21]:
initial_train_df = split_df[split_df["pool_status"] == "initial_labeled"].copy()

al_train_df = pd.concat(
    [initial_train_df, selected_al_df],
    ignore_index=True
)

print("Initial train size:", len(initial_train_df))

print("AL selected size:", len(selected_al_df))

print("New train size:", len(al_train_df))

display(al_train_df["identity_id"].value_counts())

Initial train size: 72
AL selected size: 60
New train size: 132


identity_id
G12_54    30
G12_61    29
G12_66    22
G12_62    20
G12_57    18
G8_78     13
Name: count, dtype: int64

In [22]:
al_train_dataset = PigReIDDataset(
    dataframe=al_train_df,
    transform=train_transform
)

al_train_loader = DataLoader(
    al_train_dataset,
    batch_size=16,
    shuffle=True,
    num_workers=0
)

al_batch = next(iter(al_train_loader))

In [23]:
from src.evaluation import extract_embeddings, compute_rank1_map


def evaluate_model(model):
    
    query_embeddings, query_labels, query_identity_ids, query_image_paths = extract_embeddings(
        model=model,
        dataloader=query_loader,
        device=device
    )
    
    gallery_embeddings, gallery_labels, gallery_identity_ids, gallery_image_paths = extract_embeddings(
        model=model,
        dataloader=gallery_loader,
        device=device
    )
    
    metrics = compute_rank1_map(
        query_embeddings=query_embeddings,
        query_labels=query_labels,
        gallery_embeddings=gallery_embeddings,
        gallery_labels=gallery_labels
    )
    
    return metrics

In [24]:
import importlib
import src.training
importlib.reload(src.training)

from src.training import train_reid_model

model_al, history_al = train_reid_model(
    train_df=al_train_df,
    train_transform=train_transform,
    num_classes=num_classes,
    device=device,
    epochs=10,
    batch_size=16,
    learning_rate=1e-4,
    embedding_dim=256,
    seed=42
)

Epoch 1/10:   0%|          | 0/9 [00:00<?, ?it/s]

Epoch 1: total_loss=2.0543, ce_loss=1.7839, triplet_loss=0.2704


Epoch 2/10:   0%|          | 0/9 [00:00<?, ?it/s]

Epoch 2: total_loss=1.9552, ce_loss=1.7230, triplet_loss=0.2322


Epoch 3/10:   0%|          | 0/9 [00:00<?, ?it/s]

...

Epoch 8: total_loss=1.5486, ce_loss=1.4819, triplet_loss=0.0667


Epoch 9/10:   0%|          | 0/9 [00:00<?, ?it/s]

Epoch 9: total_loss=1.5839, ce_loss=1.4726, triplet_loss=0.1113


Epoch 10/10:   0%|          | 0/9 [00:00<?, ?it/s]

Epoch 10: total_loss=1.5003, ce_loss=1.4503, triplet_loss=0.0499


In [25]:
metrics_al = evaluate_model(model_al)
print(f"After AL Rank-1: {metrics_al['rank1']:.4f}")
print(f"After AL mAP: {metrics_al['mAP']:.4f}")

After AL Rank-1: 0.3750
After AL mAP: 0.3988
